# RMT-PPAD migration NB86 - Phase P7 (validator postprocess: lane rasterizer)

**Purpose.** Verify that `MTDETRValidator.postprocess` now handles the
dict-shaped seg output produced by `LaneSegHead` and returns a
`(B, 1, 640, 640)` rasterized lane mask that the existing per-pixel
IoU/ACC metrics can consume directly.

**Acceptance (appendix-path3 sec 10.6):**
- `lanes_to_mask` produces a non-empty mask for confident priors.
- `MTDETRValidator.postprocess(...)` returns `mask.shape == (B, 1, 640, 640)`
  for a lane-only model.
- No drivable channel in the returned mask.

**Wall time:** ~2-3 min (mmcv install + model build + forward + postprocess).
Full `model.val(...)` over real BDD images is P8's job.

### Cell 1: Mount Drive, install mmcv

In [1]:
import os, sys, subprocess
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing {REPO_ROOT} -- verify Drive sync.')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

try:
    import mmcv
    print(f'[ok] mmcv: {mmcv.__version__}')
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mmcv'])
    import mmcv
    print(f'[ok] mmcv installed: {mmcv.__version__}')

import torch
print('torch', torch.__version__)
from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
[ok] mmcv installed: 2.2.0
torch 2.10.0+cu128


### Cell 2: Smoke the lane_rasterize utility

In [2]:
import sys, os
SCRIPT = 'stage2/rmt_ppad_migration/P7_validator/tools/lane_rasterize.py'
log = os.path.join(LOG_DIR, 'NB86_lane_rasterize_smoke.log')
rc = run_streaming([sys.executable, '-u', SCRIPT], log_path=log, check=False)
if rc != 0:
    raise RuntimeError(f'lane_rasterize smoke rc={rc}; see {log}')

[run_streaming] command: /usr/bin/python3 -u stage2/rmt_ppad_migration/P7_validator/tools/lane_rasterize.py
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/NB86_lane_rasterize_smoke.log
[smoke] lane_rasterize starting
[smoke] mask shape=(640, 640)  lane pixels=7918  (expect > 0)
[smoke] batched shape=(2, 1, 640, 640)  (expect (2, 1, 640, 640))
[smoke] low-confidence -> empty mask OK
[smoke] PASS
[run_streaming] return_code=0


### Cell 3: End-to-end validator postprocess check

In [3]:
import sys, os
VERIFY = 'stage2/rmt_ppad_migration/P7_validator/tools/verify_validator.py'
log = os.path.join(LOG_DIR, 'NB86_verify_validator.log')
rc = run_streaming([sys.executable, '-u', VERIFY], log_path=log, check=False)
if rc != 0:
    raise RuntimeError(f'verify_validator rc={rc}; see {log}')

print('\n[P7 result] PASS')

[run_streaming] command: /usr/bin/python3 -u stage2/rmt_ppad_migration/P7_validator/tools/verify_validator.py
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/NB86_verify_validator.log
[smoke] importing MTDETR from /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/vendor/RMT-PPAD
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[run_streaming] still running; no child output yet. This usually means the first dataloader/model step is still working.
[smoke] building model from /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/vendor/RMT-PPAD/ultralytics/cfg/models/mt-detr/rtdetr-l_bdd_clr_lane.yaml
WARNING ⚠️ no model scale passed. Assuming scale='